In [1]:
import cv2
import numpy as np

In [2]:
image_file = "test.jpg"
img = cv2.imread(image_file)

### Inverted Images

In [3]:
inverted_image = cv2.bitwise_not(img)
cv2.imwrite("inverted_image.jpg", inverted_image)

True

In [4]:
cv2.imshow("Inverted Image", inverted_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Binarization

In [5]:
def grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

In [6]:
gray_image = grayscale(img)
cv2.imwrite("gray.jpg", gray_image)

True

In [7]:
cv2.imshow("Inverted Image", gray_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## DILATION AND REMOVAL 
- it is used to thinning or bolding of the text in an image

In [8]:
def thin_font(image):
    image = cv2.bitwise_not(image)
    kernel = np.ones((2, 2), np.uint8)
    image = cv2.erode(image, kernel, iterations=1)
    return cv2.bitwise_not(image)

In [9]:
eroded_image = thin_font(img)
cv2.imwrite("thin_font.jpg", eroded_image)
cv2.imshow("new font", eroded_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
def thick_font(image):
    image = cv2.bitwise_not(image)
    kernel = np.ones((2, 2), np.uint8)
    image = cv2.dilate(image, kernel, iterations=1)
    return cv2.bitwise_not(image)

In [11]:
dilate_image = thick_font(img)
cv2.imwrite("thick_font.jpg", dilate_image)
cv2.imshow("new font", dilate_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Deskew()

In [12]:
def getSkewAngle(cvImage) -> float:
    # Prep image, copy, convert to gray scale, blur, and threshold
    newImage = cvImage.copy()
    gray = cv2.cvtColor(newImage, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # Apply dilate to merge text into meaningful lines/paragraphs.
    # Use larger kernel on X axis to merge characters into single line, cancelling out any spaces.
    # But use smaller kernel on Y axis to separate between different blocks of text
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 5))
    dilate = cv2.dilate(thresh, kernel, iterations=5)

    # Find all contours
    contours, hierarchy = cv2.findContours(dilate, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key = cv2.contourArea, reverse = True)
    for c in contours:
        rect = cv2.boundingRect(c)
        x,y,w,h = rect
        cv2.rectangle(newImage,(x,y),(x+w,y+h),(0,255,0),2)

    # Find largest contour and surround in min area box
    largestContour = contours[0]
    print (len(contours))
    minAreaRect = cv2.minAreaRect(largestContour)
    cv2.imwrite("temp/boxes.jpg", newImage)
    # Determine the angle. Convert it to the value that was originally used to obtain skewed image
    angle = minAreaRect[-1]
    if angle < -45:
        angle = 90 + angle
    return -1.0 * angle
# Rotate the image around its center
def rotateImage(cvImage, angle: float):
    newImage = cvImage.copy()
    (h, w) = newImage.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    newImage = cv2.warpAffine(newImage, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return newImage

In [13]:
# Deskew image
def deskew(cvImage):
    angle = getSkewAngle(cvImage)
    return rotateImage(cvImage, -1.0 * angle)

In [ ]:
fixed = deskew(img)
cv2.imwrite("rotated_fixed.jpg", fixed)

5


True